# Robotics Data Flywheel — complete experiment

One notebook runs setup, real-data replay validation, a development pilot, three acquisition rounds across four controls, held-out evaluation, uncertainty analysis, and video inspection.

**Start with `MODE = "quick"`** to validate the whole path. Quick mode uses real expert data and the simulator but deliberately tiny training/rollout budgets; its numbers are not performance evidence. Full mode runs five training seeds and 200 final initializations per checkpoint. No improvement is guaranteed.


In [ ]:
from pathlib import Path
import subprocess, sys
REPOSITORY = "https://github.com/0-5-blood-prince/robotics-data-flywheel.git"
REVISION = "main"  # Set to a commit SHA to reproduce a particular code version.
ROOT = Path('/content/robotics-data-flywheel')
if any(name.startswith('robot_flywheel') for name in sys.modules):
    raise RuntimeError('Restart the runtime before refreshing project code. Saved Drive checkpoints remain available.')
if not ROOT.exists():
    subprocess.run(['git', 'clone', REPOSITORY, str(ROOT)], check=True)
else:
    if not (ROOT / '.git').exists():
        raise RuntimeError('This directory came from an old ZIP. Start a fresh Colab runtime once to switch to GitHub setup.')
    changes = subprocess.check_output(['git','-C',str(ROOT),'status','--porcelain','--untracked-files=no'],text=True)
    if changes.strip(): raise RuntimeError('Local code changes found; preserve them before refreshing.')
subprocess.run(['git','-C',str(ROOT),'fetch','origin',REVISION],check=True)
subprocess.run(['git','-C',str(ROOT),'checkout','--detach','FETCH_HEAD'],check=True)
CODE_COMMIT = subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip()
print('Code commit:', CODE_COMMIT)
import os, importlib
os.chdir(ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', f'{ROOT}[simulation]'], check=True)
# Activate src in this already-running kernel: editable-install .pth files
# are normally processed only when a Python interpreter starts.
sys.path.insert(0, str(ROOT / 'src'))
importlib.invalidate_caches()
import robot_flywheel
assert Path(robot_flywheel.__file__).resolve().is_relative_to(ROOT.resolve())
print('Package ready:', robot_flywheel.__file__)

# Check the native simulator API before dataset preparation.
subprocess.run([sys.executable, '-c',
    "import gymnasium as gym, gym_pusht; e=gym.make('gym_pusht/PushT-v0'); e.reset(seed=0); e.step(e.action_space.sample()); e.close(); print('Simulator ready')"], check=True)


## Persistence and runtime
Set `USE_DRIVE=True` to preserve checkpoints across runtime replacement. Checkpoints are saved every 1,000 updates; rerunning training resumes automatically. Only load checkpoints produced by this project or another trusted source.

In [ ]:
import os, sys, json, torch, numpy as np
os.environ['SDL_VIDEODRIVER'] = 'dummy'
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = Path('/content/drive/MyDrive/robotics-data-flywheel-runs')
else:
    WORK = ROOT / 'runs'
WORK.mkdir(parents=True, exist_ok=True)
(WORK / 'code_commit.txt').write_text(CODE_COMMIT)
print({'python':sys.version,'torch':torch.__version__,'cuda':torch.cuda.is_available(),
       'device':torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})
if not torch.cuda.is_available():
    print('Switch to a GPU runtime before substantial training. CPU is suitable for tests only.')
!python -m unittest discover -s tests -v


## 1. Download and inspect the full-state expert dataset
The SHA-256 is recorded for provenance. Once verified, reuse it as `expected_sha256` on future downloads. It is not an independently authenticated upstream checksum.

In [ ]:
from robot_flywheel.prepare import prepare
from robot_flywheel.data import load_zarr
EPISODES, SOURCE = prepare(ROOT / 'data/pusht')
display(SOURCE)


In [ ]:
import matplotlib.pyplot as plt
example = EPISODES[0]
fig, ax = plt.subplots(1,2,figsize=(11,4))
ax[0].plot(example.states[:,0],example.states[:,1],label='Agent')
ax[0].plot(example.states[:,2],example.states[:,3],label='Object')
ax[0].set(xlim=(0,512),ylim=(0,512),title='Recorded demonstration',aspect='equal'); ax[0].legend()
ax[1].plot(example.actions); ax[1].set(title='Expert target positions',xlabel='Control step',ylabel='Position')
plt.show()


## 2. Compatibility gate
Replay three expert episodes. The legacy reset may shift object pose; the adapter reports both the raw and corrected reset. Missing velocities can still cause divergence. If this check fails, inspect the traces and fix compatibility before training; do not simply increase tolerances to get past the check.

In [ ]:
from robot_flywheel.rollout import check_replay
COMPATIBILITY = check_replay(EPISODES, WORK / 'compatibility', count=3)
display(COMPATIBILITY)
fig, axs = plt.subplots(1,3,figsize=(15,4))
for ax, episode in zip(axs, EPISODES[:3]):
    d = np.load(WORK / 'compatibility' / f'replay_{episode.id}.npz')
    ax.plot(d['recorded'][:,2],d['recorded'][:,3],label='Recorded object')
    ax.plot(d['simulated'][:,2],d['simulated'][:,3],label='Replayed object')
    ax.set(title=f'Episode {episode.id}',aspect='equal'); ax.legend()
plt.show()
assert COMPATIBILITY['passed'], 'Compatibility failed. Inspect report/traces before training.'


## 3. Execution mode and persistent output
Quick mode exercises every arm and all three rounds. Full mode uses five training seeds, 20,000 baseline updates, 10,000 additional updates per round, and 200 final rollouts per checkpoint. These are configurable pilot budgets, not validated convergence settings.

For full runs enable Drive persistence above. Resume requires the same frozen protocol, source data, code commit, and dependency versions. Pin `REVISION` in setup to the printed commit before a multi-session experiment. Code changes require a new study directory.

In [ ]:
from dataclasses import asdict
from robot_flywheel.experiment import Protocol, run_study
MODE = "quick"  # Change to "full" after validation and development pilot.
assert MODE in ("quick", "full")
PROTOCOL = Protocol.quick() if MODE == "quick" else Protocol()
STUDY = WORK / f"study_{MODE}_v1"
if MODE == "quick": torch.set_num_threads(1)
display(asdict(PROTOCOL))
print("Final rollouts:", len(PROTOCOL.training_seeds)*(1+4*PROTOCOL.rounds)*PROTOCOL.evaluation_count)


## 4. Development data-scaling pilot
Full mode runs 25-episode, 50-episode, and all-data references on development seeds only. The all-data reference is not an experimental initialization. If all are weak, extend training or diagnose the policy before interpreting acquisition effects. Use a new pilot directory when changing settings other than the update target.

Use these results to revise `PROTOCOL` in the following cell before freezing. Do not choose a budget based on targeted-versus-random test outcomes.

In [ ]:
from robot_flywheel.pilot import run_pilot
if MODE == "full":
    PILOT = run_pilot(EPISODES, SOURCE, COMPATIBILITY, WORK / "pilot_v1", updates=5000)
    display(PILOT)
    fig, ax = plt.subplots(figsize=(7,3))
    ax.plot([r['episodes'] for r in PILOT], [r['success_rate'] for r in PILOT], marker='o')
    ax.set(xlabel='Expert episodes',ylabel='Development success',ylim=(0,1)); plt.show()
else:
    print("Quick mode skips the convergence pilot; it validates execution only.")


## 5. Freeze the protocol
Review/edit budgets here after the pilot. No final evaluation has been run yet. Three matched acquisition arms each add the same number of complete expert episodes per round. Continued training receives the same optimizer budget but no additional episodes. Actual transition counts are also reported.

Normalization stays fixed from the initial subset. All branches inherit the same baseline weights, optimizer, and EMA state within a seed. Geometric retrieval uses failed diagnostic rollouts only; empty failure sets trigger the documented diversity fallback.

In [ ]:
from robot_flywheel.experiment import freeze_protocol
# Optional development-driven edits BEFORE the first freeze:
# PROTOCOL.baseline_updates = 40000
# PROTOCOL.initial_episodes = 50
FROZEN = freeze_protocol(PROTOCOL, SOURCE, EPISODES, STUDY)
display(FROZEN)


## 6. Train all arms and all three acquisition rounds
This cell saves after every training block and caches diagnostic rollouts. Rerun to resume. It does not run the final evaluation. All controls receive diagnostic rollouts on matched starts; only the targeted selector uses failure features.

In [ ]:
CHECKPOINTS = run_study(EPISODES, SOURCE, PROTOCOL, STUDY, COMPATIBILITY,
                        final_evaluation=False)
print("Saved checkpoints:", len(CHECKPOINTS))
display(CHECKPOINTS)


## 7. Inspect selection explanations and coverage gaps
The selector chooses complete expert episodes near failure states. It does not imitate failed policy actions. Distances above the configured support threshold flag areas the existing demonstration pool may not cover.

In [ ]:
from robot_flywheel.selection import geometry
seed = PROTOCOL.training_seeds[0]
selection_file = STUDY / f"seed_{seed}" / "targeted" / "round_1" / "selection.json"
selection = json.loads(selection_file.read_text())
display(selection)
by_id = {ep.id:ep for ep in EPISODES}
fig, ax = plt.subplots(figsize=(6,5))
for ep_id in selection['added_ids']:
    s = by_id[ep_id].states
    ax.plot(s[:,2],s[:,3],alpha=.6,label=f'Expert {ep_id}')
for ref in selection['failure_references']:
    path = selection_file.parent / 'diagnostics' / f"episode_{ref['seed']}.npz"
    with np.load(path) as d:
        point=d['states'][ref['step']]
        ax.scatter(point[2],point[3],c='black',s=12)
ax.set(xlim=(0,512),ylim=(0,512),aspect='equal',title='Selected expert paths and diagnostic failure states')
plt.show()


## 8. Held-out evaluation
Evaluate every saved checkpoint on paired initializations. The method and configuration must already be frozen. Full mode entails 13,000 rollouts, excluding diagnostics and development. Cached episode results make this resumable. Do not tune against these final results; use a new untouched seed suite for a redesigned confirmatory study.

In [ ]:
RESULTS = run_study(EPISODES, SOURCE, PROTOCOL, STUDY, COMPATIBILITY,
                    final_evaluation=True)
print("Evaluated checkpoints:", len(RESULTS))


## 9. Results, uncertainty, and data budgets
Primary contrast: targeted minus random success after round three. Crossed bootstrap resamples training seeds and shared rollout initializations while preserving arm pairing. Five seeds provide limited uncertainty resolution; a one-seed quick run provides no meaningful evidence of improvement.

Thin curves show individual training seeds. All rounds, controls, and regressions remain visible. Success is measured by the environment; maximum coverage is raw geometric overlap, not scaled reward.

In [ ]:
from robot_flywheel.report import build_report
COMPARISONS, figure = build_report(STUDY)
plt.show()
display(COMPARISONS)
import csv
with (STUDY / 'results.csv').open() as f:
    display(list(csv.DictReader(f)))


## 10. Paired rollout videos
Display the same first held-out start for baseline and final targeted policy. These are illustrative examples chosen by seed order, not selected successful episodes. Full mode records videos during evaluation; quick mode omits them to reduce CPU cost.

In [ ]:
from IPython.display import Video, display
if PROTOCOL.video_count:
    for arm, round_index in [('baseline',0),('targeted',PROTOCOL.rounds)]:
        record=next(r for r in RESULTS if r['seed']==PROTOCOL.training_seeds[0] and r['arm']==arm and r['round']==round_index)
        directory=Path(record['evaluation']).parent
        print(arm)
        display(Video(str(directory/f"episode_{PROTOCOL.evaluation_start}.mp4"),embed=True,width=400))
else:
    print("Videos disabled in quick mode; set video_count=1 before freezing a new quick study to test rendering.")


## 11. Artifact index and limitations
Exported: frozen protocol, dataset manifests, parent checkpoint hashes, selection reasons and support distances, diagnostic trajectories, model/optimizer/EMA/RNG checkpoints, final per-rollout records, CSV results, uncertainty comparisons, and plots.

Interpretation: a targeted gain must beat equal-budget random and diversity controls. Continued-training gains reveal undertraining. Episode budgets can have different transition counts; do not claim transition-level efficiency without further analysis. State-based 2D simulation does not establish real-robot or perception performance.

In [ ]:
from IPython.display import FileLink
for name in ['frozen_protocol.json','results.csv','comparisons.json','success_curves.png']:
    display(FileLink(str(STUDY/name)))
print("Artifacts:", STUDY)
print("Execution check only" if MODE == 'quick' else "Report measured results, including null or negative outcomes.")
